In [0]:
%sql
SELECT *
FROM workspace.supermarket.inventory
LIMIT 20;

In [0]:
%sql
SELECT
    s.store_name,
    p.product_name,
    p.category,
    i.quantity_on_hand
FROM workspace.supermarket.inventory i

JOIN workspace.supermarket.stores s
    ON i.store_id = s.store_id

JOIN workspace.supermarket.products p
    ON i.product_id = p.product_id

WHERE i.quantity_on_hand = 0

ORDER BY
    s.store_name,
    p.product_name;

In [0]:
%sql
SELECT
    s.store_name,
    p.product_name,
    p.category,
    i.quantity_on_hand
FROM workspace.supermarket.inventory i

JOIN workspace.supermarket.stores s
    ON i.store_id = s.store_id

JOIN workspace.supermarket.products p
    ON i.product_id = p.product_id

WHERE i.quantity_on_hand < 50

ORDER BY
    i.quantity_on_hand ASC;

In [0]:
%sql
SELECT
    s.store_name,
    p.product_name,
    p.category,
    i.quantity_on_hand,

    CASE
        WHEN i.quantity_on_hand = 0
            THEN 'OUT OF STOCK'

        WHEN i.quantity_on_hand < 50
            THEN 'LOW STOCK'

        ELSE 'NORMAL'
    END AS stock_status

FROM workspace.supermarket.inventory i

JOIN workspace.supermarket.stores s
    ON i.store_id = s.store_id

JOIN workspace.supermarket.products p
    ON i.product_id = p.product_id

ORDER BY
    i.quantity_on_hand;

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.category,

    SUM(st.quantity_sold) AS total_units_sold

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category

ORDER BY
    total_units_sold DESC;

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.category,

    SUM(st.quantity_sold) AS total_units_sold,

    DATEDIFF(
        MAX(st.sale_date),
        MIN(st.sale_date)
    ) + 1 AS selling_period_days,

    ROUND(
        SUM(st.quantity_sold) /
        NULLIF(
            DATEDIFF(
                MAX(st.sale_date),
                MIN(st.sale_date)
            ) + 1,
            0
        ),
        2
    ) AS avg_daily_sales

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category

ORDER BY
    avg_daily_sales DESC;

In [0]:
%sql
SELECT
    p.product_name,
    p.category,

    ROUND(
        SUM(st.quantity_sold) /
        NULLIF(
            DATEDIFF(
                MAX(st.sale_date),
                MIN(st.sale_date)
            ) + 1,
            0
        ),
        2
    ) AS avg_daily_sales

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category

ORDER BY
    avg_daily_sales DESC

LIMIT 10;

In [0]:
%sql
SELECT
    p.product_name,
    p.category,

    SUM(st.quantity_sold) AS total_units_sold,

    ROUND(
        SUM(st.quantity_sold) /
        NULLIF(
            DATEDIFF(
                MAX(st.sale_date),
                MIN(st.sale_date)
            ) + 1,
            0
        ),
        2
    ) AS avg_daily_sales

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category

ORDER BY
    avg_daily_sales ASC

LIMIT 10;

In [0]:
%sql
SELECT
    s.supplier_id,
    s.supplier_name,

    COUNT(po.po_id) AS delivered_orders,

    ROUND(
        AVG(
            DATEDIFF(
                po.delivery_date,
                po.order_date
            )
        ),
        2
    ) AS average_delivery_days

FROM workspace.supermarket.purchase_orders po

JOIN workspace.supermarket.suppliers s
    ON po.supplier_id = s.supplier_id

WHERE po.delivery_date IS NOT NULL

GROUP BY
    s.supplier_id,
    s.supplier_name

ORDER BY
    average_delivery_days ASC;

In [0]:
%sql
SELECT
    s.supplier_id,
    s.supplier_name,

    COUNT(po.po_id) AS delivered_orders,

    SUM(
        CASE
            WHEN DATEDIFF(
                po.delivery_date,
                po.order_date
            ) <= 7
            THEN 1
            ELSE 0
        END
    ) AS on_time_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN DATEDIFF(
                    po.delivery_date,
                    po.order_date
                ) <= 7
                THEN 1
                ELSE 0
            END
        ) /
        COUNT(po.po_id),
        2
    ) AS on_time_delivery_rate

FROM workspace.supermarket.purchase_orders po

JOIN workspace.supermarket.suppliers s
    ON po.supplier_id = s.supplier_id

WHERE po.delivery_date IS NOT NULL

GROUP BY
    s.supplier_id,
    s.supplier_name

ORDER BY
    on_time_delivery_rate DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.supplier_scorecard
USING DELTA
AS

SELECT
    s.supplier_id,
    s.supplier_name,

    COUNT(po.po_id) AS delivered_orders,

    ROUND(
        AVG(
            DATEDIFF(
                po.delivery_date,
                po.order_date
            )
        ),
        2
    ) AS average_delivery_days,

    SUM(
        CASE
            WHEN DATEDIFF(
                po.delivery_date,
                po.order_date
            ) <= 7
            THEN 1
            ELSE 0
        END
    ) AS on_time_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN DATEDIFF(
                    po.delivery_date,
                    po.order_date
                ) <= 7
                THEN 1
                ELSE 0
            END
        ) /
        COUNT(po.po_id),
        2
    ) AS on_time_delivery_rate

FROM workspace.supermarket.purchase_orders po

JOIN workspace.supermarket.suppliers s
    ON po.supplier_id = s.supplier_id

WHERE po.delivery_date IS NOT NULL

GROUP BY
    s.supplier_id,
    s.supplier_name;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.supplier_scorecard
ORDER BY on_time_delivery_rate DESC;